In [171]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.utils.read_files import load_data_if_nonempty
from maomao.hierarchical_structure.define_hierarchy_and_structure import (
    generate_sequence_id,
    normalize_sequence,
)

#### Source to mapping of toxic peptides and organism-level annotation integration
- This notebook augments the curated toxic peptide dataset with organism-level toxicity annotations derived from the original data sources.

- As input, it loads the integrated dataset produced in previous steps, including positive, negative, and ambiguous sequences, together with a curated source-level metadata table that maps each data source to its associated toxicity target (e.g., organism or biological system). All sequences are merged into a unified pivot table where each sequence retains its provenance across multiple sources.

- For each sequence, the notebook identifies which data sources provide annotations by inspecting non-missing entries in the source-specific columns. These source identifiers are then exploded into sequence–source pairs and joined with the external source-to-target mapping, allowing each peptide to be associated with one or more toxicity targets.

- The resulting data are re-aggregated into a sequence-by-target pivot table that summarizes, for each peptide, the presence of evidence across different organism or target categories. Missing associations are explicitly encoded, preserving the distinction between lack of evidence and negative annotation.

- Finally, organism-level statistics are computed and injected into the existing dataset metadata, and the sequence–target association matrix is exported as a standalone artifact. 

In [172]:
toxic_effect = "ichthyotoxic" # Change for different toxic effects (e.g., toxic, neurotoxic, hemolytic, etc.)
integration_folder = f"../../processed_data/integrating_and_cleaning_data"

- Read data

In [173]:
df_organism = (
    pd.read_excel("../../raw_data/tasks_by_source.xlsx")
    .assign(task=lambda x: x["task"].str.lower())
    .loc[lambda x: x["task"].str.contains(f"{toxic_effect}", case=False, na=False)] # Filter data sources by toxic_effect
    .iloc[:, :-4]
)

df_organism

,name source,task,toxicity target
27,AMPDB,"cytotoxic, hemolytic, platelet aggregation inh...",no information


In [174]:
df_positive = load_data_if_nonempty(key="positive", activity=toxic_effect, filename="positive.csv", path=integration_folder)
df_negative = load_data_if_nonempty(key="negative", activity=toxic_effect, filename="negative.csv", path=integration_folder)
df_ambiguo  = load_data_if_nonempty(key="ambiguous", activity=toxic_effect, filename="ambiguous_data.csv", path=integration_folder)

- Concatenate all dataset

In [175]:
df_pivote = pd.concat([df_negative, df_positive, df_ambiguo])
df_pivote = df_pivote.loc[:, ~df_pivote.columns.str.contains("unlabel")]

In [176]:
df_pivote["sequence"].unique().shape

(5,)

- Create dataset pivote

In [177]:
# Function to create the 'name source' column with the names of the columns with values ​​other than 999
def create_name_source(row):
    return [col for col in row.index[1:] if row[col] != 999]  # Excluimos la columna 'sequence'

# We apply the function row by row
df_pivote['name source'] = df_pivote.apply(create_name_source, axis=1)

# Expand the 'name source' lists into individual rows
df_exploded = df_pivote.explode('name source').reset_index(drop=True)

df_result = df_exploded[['sequence', 'name source']]

In [178]:
df_exploded = df_exploded.merge(df_organism[['name source', 'toxicity target']], on='name source', how='left')
df_exploded

,sequence,AMPDB,counts_1,counts_0,counts_unknown,positive,negative,exclusive_1,exclusive_0,percentage_0,percentage_1,name source,toxicity target
0,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,AMPDB,no information
1,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,counts_1,NaN
2,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,counts_0,NaN
3,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,counts_unknown,NaN
4,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,positive,NaN
5,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,negative,NaN
6,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,exclusive_1,NaN
7,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,exclusive_0,NaN
8,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,percentage_0,NaN
9,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,True,False,True,False,0.0,100.0,percentage_1,NaN


In [179]:
df_exploded["toxicity target"].value_counts()

toxicity target
no information    5
Name: count, dtype: int64

In [180]:
df_unique_sequences = df_exploded[['sequence', 'toxicity target']].drop_duplicates()

# Pivot the DataFrame so that the sequences are rows and the toxicity targets are columns
df_pivote_organism = df_unique_sequences.pivot_table(index='sequence', columns='toxicity target', aggfunc='size', fill_value=999)

In [181]:
df_pivote_organism

toxicity target,no information
sequence,
FIGGIISFFKRLF,1
LFGFLIKLIPSLFGALSNIGRNRNQ,1
LFGFLIPLLPHIIGAIPQVIGAIR,1
LFGFLIPLLPHLIGAIPQVIGAIR,1
NWRKILGQIASVGAGLLGSLLAGYE,1


- Working with metada

In [182]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "r") as f:
    metadata = json.load(f)

In [183]:
organism_counts = (
    df_pivote_organism
    .replace(999, 0)
    .sum()
    .astype(int)
    .to_dict()
)

In [184]:
metadata["organism_statistics"] = {"targets": organism_counts}

- Add ID column

In [185]:
if "sequence" not in df_pivote_organism.columns:
    df_pivote_organism = (
        df_pivote_organism.reset_index()
    )

df_pivote_organism = df_pivote_organism.drop(
    columns=["id"],
    errors="ignore",
)

normalized_sequences = df_pivote_organism[
    "sequence"
].map(normalize_sequence)

if normalized_sequences.isna().any():
    raise ValueError(
        "Some organism sequences could not be normalized."
    )

df_pivote_organism["sequence"] = (
    normalized_sequences
)

if df_pivote_organism["sequence"].duplicated().any():
    raise ValueError(
        "Duplicate sequences found after normalization."
    )

df_pivote_organism.insert(
    0,
    "id",
    normalized_sequences.map(
        generate_sequence_id
    ),
)

if not df_pivote_organism["id"].is_unique:
    raise ValueError(
        "Duplicate SHA-256 identifiers found."
    )

df_pivote_organism.columns.name = None

df_pivote_organism.head()

,id,sequence,no information
0,sha256_0ae112647ebefc900bfdc19f0d35a5dd9af5018...,FIGGIISFFKRLF,1
1,sha256_911187d1b6c9fb53d175badca4e92992527f10b...,LFGFLIKLIPSLFGALSNIGRNRNQ,1
2,sha256_59d6187940fb959604558f4d02ff22b7cd89c59...,LFGFLIPLLPHIIGAIPQVIGAIR,1
3,sha256_d18fee8433163c7d2c3e66785203217aeac0f6a...,LFGFLIPLLPHLIGAIPQVIGAIR,1
4,sha256_aac6eab29bd09e17936d1828acdaf551b501090...,NWRKILGQIASVGAGLLGSLLAGYE,1


- Exporting data

In [186]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [187]:
df_pivote_organism.to_csv(
    (
        f"{integration_folder}/{toxic_effect}/"
        "sequence_by_organism.csv"
    ),
    index=False,
)